# 05주 · 테일러 근사·절단오차·수렴 범위

수치해석및실습 CE273

**목표**: 수치 알고리즘·오차·수렴·안정성 이해와 방법 선택의 근거 탐구

학습 주제: 절단오차 (1) / 절단오차 (2)

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE273-numerical-analysis/week05/CE273_W05.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE273-numerical-analysis/week05)

모형·가정 확인 → 손계산·첫 갱신 → 단계별 계산 → 스텝·초기값 변경 → 오차·수렴·안정성 설명

셀 실행: Shift+Enter. 처음에는 위에서 아래로 실행. 공통 준비 셀 실행 후 각 절의 개별 실행도 가능. None 항목과 변경 셀은 직접 작성

핵심 실습: 손계산·원리와 계산식의 연결·조건 변경·해석. AI 활용은 개념 학습 후 선택 활동

[공통 학습 계획](https://github.com/steelpl/lecture/blob/main/CE273-numerical-analysis/COURSE_PLAN.md) · [학기 운영안내](https://github.com/steelpl/lecture/blob/main/CURRENT_OFFERING.md)


## 0. 실행 환경·보조 함수

필요한 버전만 설치. 업데이트한 패키지를 이미 불러온 경우 런타임 재시작 후 처음부터 실행

보조 함수의 세부 문법 암기 불필요. 입력·처리·반환값·오차 기준·실패 조건부터 확인. 아래 실습의 호출 코드와 계산식 대조


In [ ]:
# 최초 실행 시 필요한 버전만 설치. Colab에서 위에서 아래로 실행
import importlib.metadata as metadata, subprocess, sys
from packaging.version import Version
requirements = {"numpy": "1.26", "pandas": "2.2", "scipy": "1.11", "matplotlib": "3.8", "openpyxl": "3.1"}
for package, minimum in requirements.items():
    try:
        outdated = Version(metadata.version(package)) < Version(minimum)
    except metadata.PackageNotFoundError:
        outdated = True
    if outdated:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"{package}>={minimum}"])
print("실행 환경 준비 완료. 이미 불러온 패키지를 업데이트한 경우 런타임 재시작 후 처음부터 실행")

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, linalg, interpolate, integrate

def vector(values, minimum=1):
    x = np.asarray(values, dtype=float)
    if x.ndim != 1 or len(x) < minimum or not np.isfinite(x).all():
        raise ValueError("유한한 1차원 자료·최소 원소 수 확인 필요")
    return x

def relative_error(truth, estimate):
    # 기준값 0에서 상대오차를 정의하지 않음. 절대오차 별도 확인
    return np.nan if truth == 0 else abs((truth-estimate)/truth)
def series_table(kind, x, terms=10):
    if kind not in ["exp","sin","geometric"] or not np.isfinite(x) or not isinstance(terms,int) or not 1<=terms<=100:
        raise ValueError("종류·유한한 x·1~100의 정수 항 수 확인 필요")
    rows=[]; total=0.; previous=None
    truth={"exp":lambda:math.exp(x),"sin":lambda:math.sin(x),"geometric":lambda:1/(1-x)}[kind]()
    for k in range(terms):
        term = x**k/math.factorial(k) if kind=="exp" else ((-1)**k*x**(2*k+1)/math.factorial(2*k+1) if kind=="sin" else x**k)
        total += term
        step=np.nan if previous is None or total==0 else abs((total-previous)/total)*100
        rows.append({"k":k,"term":term,"estimate":total,"absolute_error":abs(truth-total),"true_relative_percent":100*relative_error(truth,total),"step_percent":step})
        previous=total
    return pd.DataFrame(rows)
def difference(f, x, h, method="central"):
    if not np.isfinite([x,h]).all() or h<=0:
        raise ValueError("유한한 x·양의 h 필요")
    if method=="forward":return (f(x+h)-f(x))/h
    if method=="backward":return (f(x)-f(x-h))/h
    if method=="central":return (f(x+h)-f(x-h))/(2*h)
    if method=="central4":return (-f(x+2*h)+8*f(x+h)-8*f(x-h)+f(x-2*h))/(12*h)
    raise ValueError("지원하는 차분 방법 확인 필요")


<a id="s01"></a>

## 전진차분과 sin 근사

**먼저 예측**: 항 하나를 더 쓰는 것과 h를 줄이는 것은 같은 조작인가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록


In [ ]:
estimate=difference(lambda x:x**3,2,.1,"forward")
sine=series_table("sin",math.pi/6,2)
print("x³ 전진차분:",estimate,"참도함수:",12.,"절대오차:",abs(estimate-12))
print(sine)
assert np.isclose(estimate,12.61) and np.isclose(sine.estimate.iloc[-1],.49967417939436376)
print("차수 n 테일러 다항식은 k=0..n의 유한합. 계수 f^(k)(a)/k!")

**조건 변경**: 전진차분 h를 절반으로 줄여 오차비 확인

예제 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**선택 심화 · AI 검토**: 함수·전개점·평가점·항 수와 다항식 차수·정의역·오차 기준 명시

**결과 검토**: 미분 가능성·Taylor 수렴·연속함수의 다항식 근사 정리를 구분

예상·중간 계산·변경 비교·독립 검산·원리 설명 기록


In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


<a id="s02"></a>

## 세 급수의 수렴 영역

**먼저 예측**: 더 많은 항이면 언제나 정답에 가까워지는가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록


In [ ]:
for kind,x in [("geometric",-.5),("geometric",.25),("geometric",1.5),("exp",.5),("sin",math.pi/6)]:
    t=series_table(kind,x,10)
    print(kind,x,"last:",t.iloc[-1][["estimate","absolute_error"]].to_dict())
convergent=series_table("geometric",.25,10).estimate.iloc[-1]
divergent=series_table("geometric",1.5,10).estimate.iloc[-1]
assert abs(convergent-1/(1-.25))<1e-5 and divergent>1
print("기하급수 합=1/(1-x)는 |x|<1에서만 성립")

**조건 변경**: 기하급수 x=.25,.99,1.5의 오차를 같은 항 수로 비교

예제 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**선택 심화 · AI 검토**: 수렴 반경·전개점·입력 x·항 수·오차표를 먼저 확인하도록 지시

**결과 검토**: 정의된 함수값과 그 Taylor 급수의 수렴은 별개

예상·중간 계산·변경 비교·독립 검산·원리 설명 기록


In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


## 주차 확인

급수·다항식·오차 한계와 수렴 범위 설명

입력 항목:

- `전진차분_x3_x2_h0_1`: 해당 계산값·항 수·오차 기준의 수치
- `sin_두항`: 해당 계산값·항 수·오차 기준의 수치
- `기하급수_x1_5는_수렴`: 판단 · Python `True`/`False`
- `연속이면_항상_자기_Taylor급수로_복원가능`: 판단 · Python `True`/`False`

공개 계산·개념 확인용 항목. 값만 복사하면 학습 완료로 판단 불가. 가정·적용 범위·검산 근거는 아래 서술과 함께 확인


In [ ]:
answers = {'전진차분_x3_x2_h0_1': None, 'sin_두항': None, '기하급수_x1_5는_수렴': None, '연속이면_항상_자기_Taylor급수로_복원가능': None}


In [ ]:
def check_answers(values):
    expected = {'전진차분_x3_x2_h0_1': 12.61, 'sin_두항': 0.49967417939436376, '기하급수_x1_5는_수렴': False, '연속이면_항상_자기_Taylor급수로_복원가능': False}
    pending, wrong = [], []
    for key, target in expected.items():
        got = values.get(key)
        if got is None:
            pending.append(key)
        elif isinstance(target, bool):
            if not isinstance(got, (bool, np.bool_)) or bool(got) != target:
                wrong.append(key)
        elif isinstance(target, str):
            if not isinstance(got, str) or got.strip().lower() != target:
                wrong.append(key)
        elif isinstance(got, (bool, np.bool_)) or not isinstance(got, (int, float, np.number)) or not np.isfinite(got) or not np.isclose(got, target, rtol=1e-6, atol=1e-8):
            wrong.append(key)
    if pending:
        print("미완료:", ", ".join(pending))
    if wrong:
        print("재검토:", ", ".join(wrong))
    if not pending and not wrong:
        print("공개 계산·개념 확인 통과. 해석·가정·AI 검토 기록은 별도 확인 필요")
    return {"pending": pending, "wrong": wrong, "passed": not pending and not wrong}

check_result = check_answers(answers)

## 원리 탐구·결과 해석 기록

1. 문제·자료·단위·모형과 가정
2. 실행 전 예측·작은 손계산과 근거
3. 기준 계산의 중간값·결과와 원리의 연결
4. 한 조건 변경 결과와 차이의 원인
5. 독립 검산·적용 조건·불확실성·한계
6. 자신의 말로 설명할 핵심 원리 한 가지

AI 활용은 선택. 사용 시 요청·의심한 출력·수정·검산 근거를 별도 기록. 글자 수 자동 채점 없음


In [ ]:
review_log = {
    "문제·자료·단위·가정": None,
    "예측·손계산 근거": None,
    "중간 계산·원리 연결": None,
    "조건 변경·차이의 원인": None,
    "검산·해석·적용 한계": None,
    "내 설명": None,
}
# 선택 활동: AI 활용 시 요청·의심한 출력·수정·검산 근거를 별도 기록
